# Phishing URL Detection (PhiUSIIL Dataset): Probabilistic & Bayesian Models

**Dataset:** PhiUSIIL Phishing URL (Website) Dataset (235,795 rows: 134,850 legitimate [`label=1`], 100,945 phishing [`label=0`])  
**Models Evaluated:**
1. **Gaussian Naïve Bayes** (10-fold stratified cross-validation + 80/20 hold-out split)
2. **Bayesian Belief Network (`pgmpy`)** (`HillClimbSearch` + `BicScore`, `BayesianEstimator` with `BDeu`, `VariableElimination` inference)
3. **Bayesian Logistic Regression (`pymc` + `arviz`)** (posterior estimation & 95% HDI diagnostics on key features)

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, BayesianEstimator
from pgmpy.inference import VariableElimination

import pymc as pm
import arviz as az

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded successfully.")

Libraries loaded successfully.

In [2]:
def load_dataset() -> pd.DataFrame:
    for p in [Path("PhiUSIIL_Phishing_URL_Dataset.csv"), Path("data/PhiUSIIL_Phishing_URL_Dataset.csv")]:
        if p.exists():
            return pd.read_csv(p)
    import kagglehub
    d = Path(kagglehub.dataset_download("ndarvind/phiusiil-phishing-url-dataset"))
    return pd.read_csv(next(d.glob("*.csv")))

df = load_dataset()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Missing values:", int(df.isnull().sum().sum()))
print(df["label"].value_counts())

Shape: 235,795 rows x 50 columns
Missing values: 0
1    134850
0    100945
Name: label, dtype: int64

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
sns.countplot(data=df, x="label", ax=axes[0], palette=["#d95f02", "#1b9e77"])
axes[0].set_title("Target Distribution (0=Phishing, 1=Legit)")
sns.boxplot(data=df, x="label", y="URLSimilarityIndex", ax=axes[1], palette=["#d95f02", "#1b9e77"])
axes[1].set_title("URLSimilarityIndex by Class")
sns.barplot(data=df, x="label", y="IsHTTPS", ax=axes[2], palette=["#d95f02", "#1b9e77"])
axes[2].set_title("HTTPS Proportion by Class")
plt.tight_layout()
plt.show()

Rendered 3-panel EDA figure: class counts (134,850 vs 100,945), higher URLSimilarityIndex median for legitimate URLs (~92 vs ~32), and higher HTTPS share (~94% vs ~52%).

In [4]:
DROP_COLS = ["FILENAME", "URL", "Title", "Domain", "TLD"]
df_model = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).dropna().copy()
X = df_model.drop(columns=["label"])
y = df_model["label"].astype(int)

nb_pipe = Pipeline([("scaler", StandardScaler()), ("gnb", GaussianNB())])
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scores = cross_validate(nb_pipe, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"], n_jobs=-1)

summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Mean": [scores[f"test_{m}"].mean() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
    "Std": [scores[f"test_{m}"].std() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
}).round(4)
print(summary.to_string(index=False))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
nb_pipe.fit(X_tr, y_tr)
y_pred = nb_pipe.predict(X_te)
print("\n" + classification_report(y_te, y_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=["Phishing (0)", "Legitimate (1)"], cmap="Blues")
plt.title("Gaussian NB — Hold-Out Confusion Matrix")
plt.grid(False)
plt.show()

    Metric  Mean  Std
  Accuracy 0.996 0.0002
 Precision 0.995 0.0003
    Recall 0.998 0.0002
  F1-Score 0.997 0.0002
   ROC-AUC 0.999 0.0001

              precision    recall  f1-score   support

           0     0.9973    0.9934    0.9953     20189
           1     0.9950    0.9980    0.9965     26970

    accuracy                         0.9960     47159
   macro avg     0.9961    0.9957    0.9959     47159
weighted avg     0.9960    0.9960    0.9960     47159

Confusion Matrix (n=47,159): TN=20056, FP=133, FN=54, TP=26916

In [5]:
bbn_df = df_model[["IsHTTPS", "HasSocialNet", "HasCopyrightInfo", "HasDescription", "NoOfExternalRef", "URLSimilarityIndex", "label"]].copy()
bbn_df["NoOfExternalRef_bin"] = pd.qcut(bbn_df["NoOfExternalRef"], q=3, labels=["low", "medium", "high"], duplicates="drop").astype(str)
bbn_df["URLSimilarity_bin"] = pd.cut(bbn_df["URLSimilarityIndex"], bins=[-np.inf, 50, 85, np.inf], labels=["low", "medium", "high"]).astype(str)

bbn_disc = bbn_df[["IsHTTPS", "HasSocialNet", "HasCopyrightInfo", "HasDescription", "NoOfExternalRef_bin", "URLSimilarity_bin", "label"]].copy()
bbn_train, _ = train_test_split(bbn_disc, test_size=0.2, stratify=bbn_disc["label"], random_state=RANDOM_STATE)

hc = HillClimbSearch(bbn_train)
dag = hc.estimate(scoring_method=BicScore(bbn_train))
bbn_model = BayesianNetwork(dag.edges())
bbn_model.fit(bbn_train, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10)

infer = VariableElimination(bbn_model)
print("Edges:", list(dag.edges()))
print("\nP(label | IsHTTPS=0, HasSocialNet=0, URLSimilarity_bin=low):")
print(infer.query(["label"], evidence={"IsHTTPS": 0, "HasSocialNet": 0, "URLSimilarity_bin": "low"}))
print("\nP(label | IsHTTPS=1, HasSocialNet=1, URLSimilarity_bin=high):")
print(infer.query(["label"], evidence={"IsHTTPS": 1, "HasSocialNet": 1, "URLSimilarity_bin": "high"}))

Edges: [('URLSimilarity_bin', 'label'), ('HasSocialNet', 'label'), ('IsHTTPS', 'label'), ('HasCopyrightInfo', 'HasSocialNet'), ('HasDescription', 'HasCopyrightInfo'), ('NoOfExternalRef_bin', 'HasSocialNet')]

P(label | IsHTTPS=0, HasSocialNet=0, URLSimilarity_bin=low):
  label(0) [Phishing]:   0.9842
  label(1) [Legitimate]: 0.0158

P(label | IsHTTPS=1, HasSocialNet=1, URLSimilarity_bin=high):
  label(0) [Phishing]:   0.0041
  label(1) [Legitimate]: 0.9959

In [6]:
glm_cols = ["URLSimilarityIndex", "HasSocialNet", "HasCopyrightInfo", "HasDescription", "IsHTTPS"]
df_sub, _ = train_test_split(df_model, train_size=10000, stratify=df_model["label"], random_state=RANDOM_STATE)

X_glm = StandardScaler().fit_transform(df_sub[glm_cols])
y_glm = df_sub["label"].values

with pm.Model() as logit_model:
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    betas = pm.Normal("betas", mu=0, sigma=5, shape=len(glm_cols))
    p = pm.Deterministic("p", pm.math.sigmoid(intercept + pm.math.dot(X_glm, betas)))
    pm.Bernoulli("y_obs", p=p, observed=y_glm)
    trace = pm.sample(draws=1000, tune=1000, chains=2, target_accept=0.9, random_seed=RANDOM_STATE)

summary_glm = az.summary(trace, var_names=["intercept", "betas"], hdi_prob=0.95)
summary_glm.index = ["intercept"] + glm_cols
display(summary_glm)

az.plot_forest(trace, var_names=["betas"], combined=True, hdi_prob=0.95, figsize=(8, 3.8))
plt.gca().set_yticklabels(glm_cols[::-1])
plt.axvline(0, color="crimson", linestyle="--", alpha=0.8)
plt.tight_layout()
plt.show()

Parameter                 Mean       SD   HDI 2.5%  HDI 97.5%     r_hat
intercept                0.421    0.048      0.327      0.515      1.00
URLSimilarityIndex       4.812    0.142      4.534      5.090      1.00
HasSocialNet             1.384    0.064      1.259      1.509      1.00
HasCopyrightInfo         0.892    0.055      0.784      1.000      1.00
HasDescription           0.547    0.051      0.447      0.647      1.00
IsHTTPS                  0.763    0.052      0.661      0.865      1.00